In [1]:
instances           = {}

import numpy as np
import pandas as pd
import copy
import time
import pickle
from util.util_load          import read_txt
from util.util_display       import plot
from util.data_indentifier   import InstanceData

from env_action.metaheu      import GeneticAlgorithm, random_population
from env_action.action_space import action_space

planning_horizon    = 480*60

directory           = 'DATA/TIGHT_DUEDATE'
ReworkProbability   = 0.03
CaseList            = ['_fixed_instance'] + [case+1 for case in range(1,48)]

PopSize             = 150
action_name         = [
                        "GA"
                        , "TS"
                        , "LFOH"
                        , "LAPH"
                        , "LAP_LFO"
                        , "LFOH_TS"
                        , "LAPH_TS"
                        , "LFOH_GA"
                        , "LAPH_GA"
                        , "CDR1"
                        , "CDR2"
                        , "CDR3"
                        # , "CDR4"
                        , "CDR5"
                        , "CDR6"
                        , "RouteChange_RightShift"
                    ]
action_id           = 3 # LAPH

In [2]:
maxtime             = 100
# CaseList = [2]

# Run only once
for CaseID in CaseList:
    print(CaseID)
    data_path = f"{directory}/Case{CaseID}_480.txt"
    # data_path = "DATA/jobs_small.txt"
    J, I, K, p_ijk, h_ijk,   \
    d_j, n_j, MC_ji, n_MC_ji,\
    OperationPool          = read_txt(data_path)

    S_k                    = np.zeros((K))
    S_j                    = np.zeros((J))
    n_ops_left_j           = copy.deepcopy(n_j)
    MB_record = {}
    t                      = 0
    JSet                   = list(range(J))
    OJSet                  = [[] for _ in range(J)]
    for j in JSet:
        OJSet[j]           = [i for i in range(int(n_j[j]))]

    T_cur = 0
    Tard_job = 0
    Oij_on_machine = 0
    affected_Oij = 0
    NewJobList = 0
    CT_k = 0
    X_ijk = 0
    S_ij = 0
    C_ij = 0
    C_j = 0
    re = 0

    action_method 						         = action_space(J, I, K, p_ijk, h_ijk, d_j, n_j, 
                                                                MC_ji, n_MC_ji, n_ops_left_j, OperationPool, S_k, S_j, 
                                                                JSet, OJSet, affected_Oij, 
                                                                t, X_ijk, S_ij, C_ij, C_j, CT_k, T_cur, Tard_job,
                                                                NewJobList, PopSize, maxtime, re)
    
    # print(action_name[action_id])

    reschedule							         = action_method[action_id]
    GBest, X_ijk, S_ij, C_ij, C_j                = reschedule()
    # fig1 = plot(J, K, n_j, X_ijk, S_ij, C_ij, MB_record, t)
    # display(fig1)
    remaining_info_file = f'{directory}/Case{CaseID}_{planning_horizon // 60}_InfoNewJob.pkl'
    with open(remaining_info_file, 'rb') as f:
        remaining_batches = pickle.load(f)

    new_job_indices = [comp_id for comp_id, qty in remaining_batches.items() for _ in range(qty)]
    instances[CaseID]             = InstanceData(J, I, X_ijk, S_ij, C_ij, C_j, p_ijk, h_ijk, 
                                                    d_j, n_j, MC_ji, n_MC_ji, OperationPool, new_job_indices)


_fixed_instance
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48


In [3]:
# Store data
with open(f'{directory}/pickle_instances_480.pkl', 'wb') as f:
    pickle.dump(instances, f)

In [4]:
from util.util_load        import read_scenario
from util.data_indentifier import ScenarioData

import pickle
K = 30
scenarios           = {}
critical_machines   = {5, 6, 7, 8, 9, 10, 11, 12, 13, 21, 22, 26, 27}
ScenarioList        = ['_fixed_scenario', 'A', 'B', 'C', 'D', 'E', 'F', 'G']

for ScenarioID in ScenarioList:
    scenario_path = f"{directory}/Scenario{ScenarioID}_480.txt"
    JA_event, MB_event     = read_scenario(scenario_path, K, critical_machines)
    scenarios[ScenarioID]  = ScenarioData(JA_event, MB_event)

with open(f'{directory}/pickle_scenarios_480.pkl', 'wb') as f:
    pickle.dump(scenarios, f)